In [0]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import when ,datediff
from pyspark.sql.functions import to_timestamp
from pyspark.sql.functions import col
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, FloatType, DoubleType, BooleanType, DateType, TimestampType
)
spark = SparkSession.builder.appName("ecommerceData").getOrCreate()


In [0]:
df_customers = spark.table("de_mini_project.01_bronze_schema.olist_customers_dataset")
df_geolocation = spark.read.table("de_mini_project.01_bronze_schema.olist_geolocation_dataset")
df_orders = spark.read.table("de_mini_project.01_bronze_schema.olist_orders_dataset")
df_order_items = spark.read.table("de_mini_project.01_bronze_schema.olist_order_items_dataset")
df_products = spark.read.table("de_mini_project.01_bronze_schema.olist_products_dataset")
df_sellers = spark.read.table("de_mini_project.01_bronze_schema.olist_sellers_dataset")
df_payment = spark.read.table("de_mini_project.01_bronze_schema.olist_order_payments_dataset")
df_reviews = spark.read.table("de_mini_project.01_bronze_schema.olist_order_reviews")
df_category = spark.read.table("de_mini_project.01_bronze_schema.product_category_name_translation")
     

In [0]:
df_orders = spark.read.table("de_mini_project.01_bronze_schema.olist_orders_dataset")
display(df_orders)

##Test Displaying

In [0]:
# display(df_customers.limit(5))
# display(df_geolocation.limit(5))
# display(df_orders.limit(5))
# display(df_order_items.limit(5))
# display(df_products.limit(5))
# display(df_sellers.limit(5))
# display(df_payment.limit(5))
# display(df_reviews.limit(5))
# display(df_category.limit(5))

### 1.Customers

In [0]:
import pyspark.sql.functions as F
df_customers = df_customers.dropDuplicates()
df_customers = df_customers.withColumnRenamed('customer_zip_code_prefix','zip_code_prefix')
df_customers = df_customers.withColumnRenamed('customer_city','city')
df_customers = df_customers.withColumnRenamed('customer_state','state')
df_customers = df_customers.withColumnRenamed('customer_unique_id','unique_id')
df_customers = df_customers.withColumnRenamed("city","city")
df_customers = df_customers.withColumn("state",F.lower(F.col("state")))

In [0]:
display(df_customers)

### 2.Geolocation

In [0]:
df_geolocation = df_geolocation.dropDuplicates()
df_geolocation = df_geolocation.withColumnRenamed('geolocation_zip_code_prefix','zip_code_prefix').withColumnRenamed('geolocation_city','city').withColumnRenamed('geolocation_state','state').withColumnRenamed('geolocation_lat','latitude').withColumnRenamed('geolocation_lng','longitude')
df_geolocation = df_geolocation.withColumn("city",F.initcap(F.col("city")))

### 3.Orders

In [0]:
df_orders = df_orders.withColumnRenamed('order_purchase_timestamp','order_purchase_timestamp').withColumnRenamed('order_approved_at','order_approved_at').withColumnRenamed('order_delivered_carrier_date','order_delivered_carrier_date').withColumnRenamed('order_delivered_customer_date','order_delivered_customer_date').withColumnRenamed('order_estimated_delivery_date','order_estimated_delivery_date')
from pyspark.sql.functions import to_timestamp
 
df_orders = spark.read.table("de_mini_project.01_bronze_schema.olist_orders_dataset")
 
df_orders = df_orders.dropDuplicates()
 
# Standardizing all timestamp columns
timestamp_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
 
for col_name in timestamp_cols:
    df_orders = df_orders.withColumn(col_name, to_timestamp(col(col_name)))
 
# Standardizingorder_status text
df_orders = df_orders.withColumn("order_status", F.lower(F.trim(col("order_status"))))
 

display(df_orders.limit(100))


In [0]:
df_orders.printSchema()

###Orders - Ignoring the Columns without order_purchase_TimeStamps


In [0]:
df_orders = df_orders.filter(col("order_purchase_timestamp").isNotNull())

### Orders - Transformation -Adding total Delivery days

In [0]:
from pyspark.sql.functions import col, when, datediff, lit, coalesce, to_timestamp

df_orders = df_orders.fillna("unknown")
df_orders = df_orders.replace("nan", "unknown", subset=None)
df_orders = df_orders.dropDuplicates()

timestamp_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col_name in timestamp_cols:
    df_orders = df_orders.withColumn(
        col_name,
        to_timestamp(col(col_name))
    )

# Convert specified timestamp columns to string and replace nulls with "unknown"
for col_name in [
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]:
    df_orders = df_orders.withColumn(
        col_name,
        when(col(col_name).isNull(), lit("unknown"))
        .otherwise(col(col_name).cast("string"))
    )

# values to be null
order_status_zero_days = ["processing", "shipped", "unavailable", "invoiced", "canceled"]

df_orders = df_orders.withColumn(
    "delivery_days",
    when(
        col("order_status").isin(order_status_zero_days),
        lit(0)
    ).when(
        (col("order_status") == "delivered") &
        (col("order_delivered_customer_date") != "unknown") &
        (col("order_purchase_timestamp").isNotNull()),
        datediff(
            to_timestamp(col("order_delivered_customer_date")),
            col("order_purchase_timestamp")
        )
    ).otherwise(lit(None).cast("bigint"))
)

# delivery_before_purchase (1 if delivered before purchase, else 0)
df_orders = df_orders.withColumn(
    "delivery_before_purchase",
    when(
        (col("order_delivered_customer_date") != "unknown") &
        (col("order_purchase_timestamp").isNotNull()) &
        (to_timestamp(col("order_delivered_customer_date")) < col("order_purchase_timestamp")),
        1
    ).otherwise(0)
)

# is_valid_delivery logic: 0 if delivery_days is null or order_delivered_carrier_date/order_delivered_customer_date is unknown, else 1 if delivered after purchase
df_orders = df_orders.withColumn(
    "is_valid_delivery",
    when(
        col("delivery_days").isNull() |
        (col("order_delivered_carrier_date") == "unknown") |
        (col("order_delivered_customer_date") == "unknown"),
        0
    ).otherwise(
        when(
            (col("order_delivered_customer_date") == "unknown") |
            col("order_purchase_timestamp").isNull() |
            (to_timestamp(col("order_delivered_customer_date")) >= col("order_purchase_timestamp")),
            1
        ).otherwise(0)
    )
)

# is_late_delivery: 1 if delivered after estimated delivery date, else 0
df_orders = df_orders.withColumn(
    "is_late_delivery",
    when(
        (col("order_delivered_customer_date") != "unknown") &
        (col("order_estimated_delivery_date") != "unknown") &
        (to_timestamp(col("order_delivered_customer_date")) > to_timestamp(col("order_estimated_delivery_date"))),
        1
    ).otherwise(0)
)

display(df_orders)

In [0]:
display(df_orders)

### 4 Order_items

In [0]:
# display(df_order_items)
df_order_items = df_order_items.dropDuplicates()
df_order_items = df_order_items.filter(col("order_id").isNotNull())

df_order_items = df_order_items.withColumn(
    "price",
    when(col("price") == 0, F.lit(0.0)).otherwise(col("price"))
)
df_order_items = df_order_items.withColumn(
    "freight_value",
    when(col("freight_value") == 0, F.lit(0.0)).otherwise(col("freight_value").cast("double"))
)
df_order_items = df_order_items.withColumn(
    "shipping_limit_date_only",
    F.to_date(col("shipping_limit_date"))
)
df_order_items = df_order_items.withColumn(
    "shipping_limit_time_only",
    F.date_format(col("shipping_limit_date"), "HH:mm:ss")
)
display(df_order_items.limit(1000))

### 5 - Payments

In [0]:
# Do this for all the tables 
df_payment = df_payment.filter(col("order_id").isNotNull())
df_payment = df_payment.dropDuplicates()
df_payment  = df_payment.fillna({"payment_type": "unkown"})
df_payment = df_payment.withColumn(
    "payment_value",
    when(col("payment_value") == 0, F.lit(0.0)).otherwise(col("payment_value").cast("double")))
# display(df_payment.limit(1000))
from pyspark.sql.functions import sum, count, collect_list

df_payment = spark.read.table("de_mini_project.01_bronze_schema.olist_order_payments_dataset")

df_payment = df_payment.dropDuplicates()

df_payment = df_payment.fillna({"payment_type": "unknown"})

# Saving the raw cleaned version 
df_payment.write.format("delta").mode("overwrite") \
    .saveAsTable("de_mini_project.02_silver_schema.payments")

print("payment saved —", df_payment.count(), "rows")

# Creating an aggregated view per order 
df_payment_agg = df_payment.groupBy("order_id").agg(
    sum("payment_value").alias("total_payment_value"),
    count("payment_sequential").alias("payment_count"),
    collect_list("payment_type").alias("payment_types_used")
)

df_payment = df_payment.join(df_payment_agg, on="order_id", how="left")

display(df_payment)

### 6 - Reviews

In [0]:
df_reviews = df_reviews.filter(col("order_id").isNotNull())
df_reviews = df_reviews.dropDuplicates()
df_reviews = df_reviews.withColumn(
    "review_score",
    when(col("review_score") == 0, F.lit(0.0)).otherwise(col("review_score").cast("double"))
)
df_reviews = df_reviews.withColumn(
    "review_comment_title",
    when(col("review_comment_title") == "nan", F.lit("unknown"))
    .otherwise(
        coalesce(
            col("review_comment_title").cast("string"),
            F.lit("unknown")
        )
    )
)
df_reviews = df_reviews.withColumn(
    "review_comment_message",
    when(col("review_comment_message") == "nan", F.lit("unknown"))
    .otherwise(
        coalesce(
            col("review_comment_message").cast("string"),
            F.lit("unknown")
        )
    )
)

display(df_reviews)

### 7 Products

In [0]:
display(df_products)

In [0]:
df_products = df_products.withColumnRenamed("product_name_lenght","product_name_length").withColumnRenamed("product_description_lenght","product_description_length")
df_products = df_products.filter(col("product_id").isNotNull())
df_products = df_products.dropDuplicates()
df_products = df_products.withColumn(
    "product_weight_g",
    when(col("product_weight_g") == 0, F.lit(0.0)).otherwise(col("product_weight_g").cast("double"))
)
df_products = df_products.withColumn(
    "product_length_cm",
    when(col("product_length_cm") == 0, F.lit(0.0)).otherwise(col("product_length_cm").cast("double"))
)
df_products = df_products.withColumn(
    "product_height_cm",
    when(col("product_height_cm") == 0, F.lit(0.0)).otherwise(col("product_height_cm").cast("double"))
)
df_products = df_products.withColumn(
    "product_width_cm",
    when(col("product_width_cm") == 0, F.lit(0.0)).otherwise(col("product_width_cm").cast("double"))
)
df_products = df_products.withColumn(
    "product_photos_qty",
    when(col("product_photos_qty") == 0, F.lit(0.0)).otherwise(col("product_photos_qty").cast("double"))
)
df_products = df_products.withColumn(
    "product_name_length",
    when(col("product_name_length") == 0, F.lit(0)).otherwise(col("product_name_length").cast("int"))
)
df_products = df_products.withColumn(
    "product_description_length",
    when(col("product_description_length") == 0, F.lit(0.0)).otherwise(col("product_description_length").cast("double"))
)

df_products = df_products.withColumn(
    "product_category_name",
    coalesce(
        col("product_category_name").cast("string"),
        F.lit("unknown")
    )
)
df_products = df_products.withColumn(
    "product_description_length",
    coalesce(
        col("product_description_length").cast("double"),
        F.lit(0)
    )
)
df_products = df_products.withColumn(
    "product_photos_qty",
    coalesce(
        col("product_photos_qty").cast("int"),
        F.lit(0)
    )
)
df_products = df_products.withColumn(
    "product_name_length",
    coalesce(
        col("product_name_length").cast("int"),
        F.lit(0)
    )
)
display(df_products)

### 8 Sellers

In [0]:
df_sellers = df_sellers.filter(col("seller_id").isNotNull())
df_sellers = df_sellers.dropDuplicates()
df_sellers = df_sellers.withColumn(
    "seller_zip_code_prefix",
    when(col("seller_zip_code_prefix") == 0, F.lit(0)).otherwise(col("seller_zip_code_prefix").cast("int"))
)
df_sellers = df_sellers.fillna({
  "seller_city": "unknown",
  "seller_state": "unknown"
})
df_sellers = df_sellers.withColumn("seller_city", F.lower(col("seller_city")))
df_sellers = df_sellers.withColumn("seller_state", F.lower(col("seller_state")))
display(df_sellers)

### 9 Category

In [0]:
df_category = df_category.filter(col("product_category_name").isNotNull())
df_category = df_category.dropDuplicates()
df_category = df_category.withColumnRenamed("product_category_name","product_category_name")
df_category = df_category.withColumnRenamed("product_category_name_english","product_category_name_english")
display(df_category)

## REFERENTIAL INTEGRITY CHECKING

In [0]:
print("   REFERENTIAL INTEGRITY CHECK — SILVER LAYER")
#1 Orders -> Customers 
orphan_orders = df_orders.join(df_customers, "customer_id", "left_anti")
orphan_orders_count = orphan_orders.count()

#2 Order Items -> Orders 
orphan_items_order = df_order_items.join(df_orders, "order_id", "left_anti")
orphan_items_order_count = orphan_items_order.count()

# 3 Order Items -> Products
orphan_items_product = df_order_items.join(df_products, "product_id", "left_anti")
orphan_items_product_count = orphan_items_product.count()

# 4 Order Items -> Sellers 
orphan_items_seller = df_order_items.join(df_sellers, "seller_id", "left_anti")
orphan_items_seller_count = orphan_items_seller.count()

#  5 Payments ->Orders 
orphan_payments = df_payment.join(df_orders, "order_id", "left_anti")
orphan_payments_count = orphan_payments.count()

#  6 Reviews -> Orders 
orphan_reviews = df_reviews.join(df_orders, "order_id", "left_anti")
orphan_reviews_count = orphan_reviews.count()

# Print Report 
print(f"\nOrders → Customers  orphans : {orphan_orders_count}")
print(f"  Items    → Orders     orphans : {orphan_items_order_count}")
print(f"  Items    → Products   orphans : {orphan_items_product_count}")
print(f"  Items    → Sellers    orphans : {orphan_items_seller_count}")
print(f"  Payments → Orders     orphans : {orphan_payments_count}")
print(f"  Reviews  → Orders     orphans : {orphan_reviews_count}")

#Final checking
all_checks = [
    orphan_orders_count,
    orphan_items_order_count,
    orphan_items_product_count,
    orphan_items_seller_count,
    orphan_payments_count,
    orphan_reviews_count
]

if all(v == 0 for v in all_checks):
    print("ARI CHECKS are  PASSED")
    print("Safe to write to Silver catalog")
else:
    print("Orphan records found - see numbers above")

### Pushing all transformed code into Silver Catalog

In [0]:
df_category.write.format("delta").mode("overwrite").saveAsTable("de_mini_project.02_silver_schema.category")
df_sellers.write.format("delta").mode("overwrite").saveAsTable("de_mini_project.02_silver_schema.sellers")
df_products.write.format("delta").mode("overwrite").saveAsTable("de_mini_project.02_silver_schema.products")
df_reviews.write.format("delta").mode("overwrite").saveAsTable("de_mini_project.02_silver_schema.reviews")
df_payment.write.format("delta").mode("overwrite").saveAsTable("de_mini_project.02_silver_schema.payments")
df_order_items.write.format("delta").mode("overwrite").saveAsTable("de_mini_project.02_silver_schema.order_items")
df_orders.write.option("overwriteSchema", "true").format("delta").mode("overwrite").saveAsTable("de_mini_project.02_silver_schema.orders")
df_geolocation.write.format("delta").mode("overwrite").saveAsTable("de_mini_project.02_silver_schema.geolocation")
df_customers.write.format("delta").mode("overwrite").saveAsTable("de_mini_project.02_silver_schema.customers")